In [2]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. LOAD, TRACK OVERLOADS, AND IMPUTE
def load_swiss_roll_csv(file_path):
    header_idx = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if any(key in line for key in ['Scan Num', '101 (', 'Scan Swee']):
                header_idx = idx
                break
                
    df = pd.read_csv(file_path, skiprows=header_idx)
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    time_cols = [col for col in df.columns if 'time' in col.lower() or 'swee' in col.lower()]
    df['Timestamp'] = df[time_cols[0]] if time_cols else df.index
    
    sensor_cols = [col for col in df.columns if '°C' in col]
    
    for col in sensor_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Track the exact locations of the hardware overloads
        overload_col = f"overload_flag_{col}"
        df[overload_col] = (df[col].abs() > 10000) | (df[col] == np.inf)
        
        # Scrub for the neural network
        df.loc[df[overload_col], col] = np.nan
        
    # Impute gaps to maintain sequence integrity for the LSTM
    df[sensor_cols] = df[sensor_cols].fillna(method='ffill').fillna(method='bfill')
        
    return df, sensor_cols

# 2. CREATE SLIDING TIME WINDOWS
def create_sequences(X, timestamps, time_steps):
    Xs, ts = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X.iloc[i:(i + time_steps)].values)
        ts.append(timestamps.iloc[i + time_steps - 1])
    return np.array(Xs), np.array(ts)

# 3. PYTORCH LSTM AUTOENCODER ARCHITECTURE
class LSTMAutoencoder(nn.Module):
    def __init__(self, num_features, hidden_size=64):
        super(LSTMAutoencoder, self).__init__()
        self.hidden_size = hidden_size
        self.encoder_lstm = nn.LSTM(input_size=num_features, hidden_size=hidden_size, batch_first=True)
        self.decoder_lstm = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, num_features)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        _, (hidden_state, _) = self.encoder_lstm(x)
        last_hidden_state = hidden_state[-1]
        repeated_hidden = last_hidden_state.unsqueeze(1).repeat(1, seq_len, 1)
        decoded, _ = self.decoder_lstm(repeated_hidden)
        reconstructed = self.output_layer(decoded)
        return reconstructed

# 4. TRAINING LOOP
def train_model(model, train_loader, num_epochs=35, learning_rate=0.001):
    print("Beginning PyTorch training loop on 13 channel data")
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for batch_x in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x[0])
            loss = criterion(outputs, batch_x[0])
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
    print("Training complete.")
    return model

# 5. UNSUPERVISED SCORING
def evaluate_and_score(model, X_seq_tensor, df_timestamps, contamination=0.03):
    model.eval()
    with torch.no_grad():
        reconstructed = model(X_seq_tensor)
        
    X_seq_np = X_seq_tensor.numpy()
    reconstructed_np = reconstructed.numpy()
    
    mae_loss = np.mean(np.abs(reconstructed_np - X_seq_np), axis=(1, 2))
    
    results_df = pd.DataFrame({
        'Timestamp': df_timestamps,
        'Reconstruction_Error_MAE': mae_loss
    })
    
    threshold = np.percentile(mae_loss, (1 - contamination) * 100)
    results_df['Is_Anomaly'] = results_df['Reconstruction_Error_MAE'] > threshold
    
    print("**************************************************")
    print(" 13 CHANNEL SWISS ROLL AUTOENCODER METRICS")
    print("**************************************************")
    print(f"Total Sequences Analyzed : {len(results_df)}")
    print(f"Mean Reconstruction Error: {np.mean(mae_loss):.4f}")
    print(f"Max Reconstruction Error : {np.max(mae_loss):.4f}")
    print(f"Decision Threshold       : {threshold:.4f}")
    print(f"Total Anomalies Flagged  : {results_df['Is_Anomaly'].sum()}")
    print("**************************************************\n")
    
    return results_df, threshold

# 6. DYNAMIC 14 ROW VISUALIZATION WITH OVERLOAD MARKERS
def plot_results(df_original, results_df, sensor_cols, threshold, file_name):
    df_merged = pd.merge(df_original, results_df, on='Timestamp', how='inner')
    anomalies = df_merged[df_merged['Is_Anomaly'] == True]
    
    num_sensors = len(sensor_cols)
    fig = make_subplots(
        rows=num_sensors + 1, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.02,
        subplot_titles=["<b>Reconstruction Error MAE Data Points</b>"] + [f"Ch: {col}" for col in sensor_cols]
    )
    
    normal_points = df_merged[df_merged['Is_Anomaly'] == False]
    
    # Plot MAE Data Points
    fig.add_trace(
        go.Scatter(
            x=normal_points['Timestamp'], y=normal_points['Reconstruction_Error_MAE'],
            mode='markers', name='Normal MAE', marker=dict(color='blue', size=4, opacity=0.6)
        ), row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=anomalies['Timestamp'], y=anomalies['Reconstruction_Error_MAE'],
            mode='markers', name='Anomalous MAE', marker=dict(color='red', size=6, symbol='x')
        ), row=1, col=1
    )
    fig.add_hline(y=threshold, line_dash="dash", line_color="black", row=1, col=1)
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896']
    
    for i, col in enumerate(sensor_cols):
        row_idx = i + 2
        
        # Plot continuous sequence
        fig.add_trace(
            go.Scatter(
                x=df_merged['Timestamp'], y=df_merged[col], mode='lines', 
                line=dict(color=colors[i % len(colors)], width=1.5), showlegend=False
            ), row=row_idx, col=1
        )
        
        # Plot sequence anomalies (Red Diamonds)
        fig.add_trace(
            go.Scatter(
                x=anomalies['Timestamp'], y=anomalies[col], mode='markers', 
                marker=dict(color='red', size=6, symbol='diamond'), showlegend=False
            ), row=row_idx, col=1
        )
        
        # Plot exact hardware overload locations (Orange X)
        overload_col = f"overload_flag_{col}"
        hardware_faults = df_merged[df_merged[overload_col] == True]
        
        if not hardware_faults.empty:
            fig.add_trace(
                go.Scatter(
                    x=hardware_faults['Timestamp'], y=hardware_faults[col], mode='markers',
                    marker=dict(color='orange', size=10, symbol='x', line=dict(width=2, color='black')), 
                    name="Hardware Fault", showlegend=False
                ), row=row_idx, col=1
            )
            
        fig.update_yaxes(title_text="C", title_font=dict(size=10), row=row_idx, col=1)

    fig.update_layout(
        title=f"13 Channel Swiss Roll Sequence Reconstruction: {file_name}",
        height=300 + (130 * num_sensors),
        width=1250,
        hovermode='x unified'
    )
    fig.show()

# 7. EXECUTION
file_path = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\Swiss Roll\1781333575_2slpm 0.csv"

if os.path.exists(file_path):
    df, sensors = load_swiss_roll_csv(file_path)
    
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df[sensors]), columns=sensors)
    
    TIME_STEPS = 10
    X_seq, timestamps_seq = create_sequences(df_scaled, df['Timestamp'], TIME_STEPS)
    X_tensor = torch.tensor(X_seq, dtype=torch.float32)
    
    dataset = TensorDataset(X_tensor)
    train_loader = DataLoader(dataset, batch_size=16, shuffle=False)
    
    num_features = len(sensors)
    pytorch_model = LSTMAutoencoder(num_features=num_features, hidden_size=64)
    trained_model = train_model(pytorch_model, train_loader, num_epochs=35)
    
    results, decision_threshold = evaluate_and_score(trained_model, X_tensor, timestamps_seq, contamination=0.03)
    
    plot_results(df, results, sensors, decision_threshold, os.path.basename(file_path))
else:
    print(f"File not found: {file_path}")

C:\Users\hari7\AppData\Local\Temp\ipykernel_14948\372599599.py:41: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[sensor_cols] = df[sensor_cols].fillna(method='ffill').fillna(method='bfill')
c:\Users\hari7\Documents\Semester 3\Projects\mass_panrom\Lib\site-packages\sklearn\utils\extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
c:\Users\hari7\Documents\Semester 3\Projects\mass_panrom\Lib\site-packages\sklearn\utils\extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
c:\Users\hari7\Documents\Semester 3\Projects\mass_panrom\Lib\site-packages\sklearn\utils\extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


Beginning PyTorch training loop on 13 channel data
Training complete.
**************************************************
 13 CHANNEL SWISS ROLL AUTOENCODER METRICS
**************************************************
Total Sequences Analyzed : 266
Mean Reconstruction Error: nan
Max Reconstruction Error : nan
Decision Threshold       : nan
Total Anomalies Flagged  : 0
**************************************************

